In [8]:
# Cài đặt Git LFS để tải file model nặng
!apt-get install git-lfs -y
!git lfs install

# Clone repository từ Hugging Face Space
!git clone https://huggingface.co/spaces/Nick088/Real-ESRGAN_Pytorch
%cd Real-ESRGAN_Pytorch

zsh:1: command not found: apt-get
git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	refs
Cloning into 'Real-ESRGAN_Pytorch'...
remote: Enumerating objects: 306, done.
remote: Total 306 (delta 0), reused 0 (delta 0), pack-reused 306 (from 1)
Receiving objects: 100% (306/306), 246.26 KiB | 14.49 MiB/s, done.
Resolving deltas: 100% (166/166), done.
/Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch/Real-ESRGAN_Pytorch


In [9]:
# Cài đặt các thư viện từ file requirements
!pip install -r requirements.txt
# Cài đặt thêm Gradio nếu file requirements chưa có
!pip install gradio

  Using cached gradio-4.28.3-py3-none-any.whl.metadata (15 kB)
  Using cached gradio_client-0.16.0-py3-none-any.whl.metadata (7.1 kB)
Using cached gradio-4.28.3-py3-none-any.whl (12.2 MB)
Using cached gradio_client-0.16.0-py3-none-any.whl (314 kB)
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.5.0
    Uninstalling gradio_client-2.5.0:
      Successfully uninstalled gradio_client-2.5.0
  Attempting uninstall: gradio
    Found existing installation: gradio 6.13.0
    Uninstalling gradio-6.13.0:
      Successfully uninstalled gradio-6.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gradio]2m1/2 [gradio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 0.16.0 which is incompatible.


In [10]:
!pip install --upgrade huggingface_hub gradio

  Using cached gradio-6.13.0-py3-none-any.whl.metadata (17 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
Using cached gradio-6.13.0-py3-none-any.whl (19.7 MB)
Using cached gradio_client-2.5.0-py3-none-any.whl (59 kB)
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 0.16.0
    Uninstalling gradio_client-0.16.0:
      Successfully uninstalled gradio_client-0.16.0
  Attempting uninstall: gradio
    Found existing installation: gradio 4.28.3
    Uninstalling gradio-4.28.3:
      Successfully uninstalled gradio-4.28.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gradio]2m1/2 [gradio]


In [19]:
from pathlib import Path
import re

repo = Path('/Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch')
app_path = repo / 'app.py'
infer_path = repo / 'infer.py'

# --- 1) Enforce app.py as image-only Gradio UI and correct output type ---
app_text = '''import gradio as gr

from infer import infer_image

input_image = gr.Image(type='pil', label='Input Image')
input_model_image = gr.Radio([('x2', 2), ('x4', 4), ('x8', 8)], type="value", value=4, label="Model Upscale/Enhance Type")
submit_image_button = gr.Button('Submit')
output_image = gr.Image(type="pil", label="Output Image")

tab_img = gr.Interface(
    fn=infer_image,
    inputs=[input_image, input_model_image],
    outputs=output_image,
    title="Real-ESRGAN Pytorch",
)


demo = tab_img

demo.launch(debug=True, show_error=True, share=True)
'''
app_path.write_text(app_text, encoding='utf-8')
print(f'Updated app: {app_path}')

# --- 2) Ensure infer.py works without Hugging Face spaces package ---
infer_text = infer_path.read_text(encoding='utf-8')

if 'import spaces' in infer_text and 'class _SpacesShim' not in infer_text:
    infer_text = infer_text.replace(
        'import spaces',
        """try:
    import spaces
except ModuleNotFoundError:
    class _SpacesShim:
        @staticmethod
        def GPU(duration=None):
            # Local fallback when Hugging Face Spaces runtime is unavailable.
            def _decorator(func):
                return func

            return _decorator

    spaces = _SpacesShim()""",
        1,
    )
    infer_path.write_text(infer_text, encoding='utf-8')
    print(f'Updated infer: {infer_path}')
else:
    print('infer.py already has spaces fallback (no change needed).')

print('Done. Run the app launch cell next.')

Updated app: /Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch/app.py
infer.py already has spaces fallback (no change needed).
Done. Run the app launch cell next.


In [ ]:
# Cell 5: Patch stable app + synchronized zoom/pan on 3 previews
%cd /kaggle/working/Real-ESRGAN_Pytorch

from pathlib import Path
import re

# 1) Keep infer.py safe for environments without `spaces`
infer_path = Path('infer.py')
infer_text = infer_path.read_text(encoding='utf-8')

if 'import spaces' in infer_text and '_SpacesShim' not in infer_text:
    infer_text = infer_text.replace(
        'import spaces',
        """try:
    import spaces
except ModuleNotFoundError:
    class _SpacesShim:
        @staticmethod
        def GPU(duration=None):
            def _decorator(func):
                return func
            return _decorator

    spaces = _SpacesShim()""",
        1,
    )

# In Kaggle we do not need Spaces decorators.
infer_text = re.sub(r'^@spaces\.GPU\(.*\)\n', '', infer_text, flags=re.MULTILINE)
infer_path.write_text(infer_text, encoding='utf-8')
print('infer.py patched')

# 2) Write stable Gradio app + synchronized zoom/pan for input/output/groundtruth
app_code = '''import gradio as gr
from PIL import Image

from infer import infer_image


def infer_image_for_ui(img: Image.Image, size_modifier: int, gt_img: Image.Image):
    if img is None:
        return None, None, gt_img
    result = infer_image(img, size_modifier)
    return img, result, gt_img


custom_css = """
/* Keep the same frame height and behavior for all three previews */
#input_preview, #output_image, #groundtruth_preview {
  min-height: 340px;
}
#input_preview .image-container, #output_image .image-container, #groundtruth_preview .image-container {
  height: 340px;
  overflow: hidden;
  touch-action: none;
  background: #111;
}
#input_preview img, #output_image img, #groundtruth_preview img {
  transform-origin: 0 0;
  user-select: none;
  -webkit-user-drag: none;
}
"""

sync_js = r"""
() => {
  try {
    if (window.__esr_sync_bound) return;
    window.__esr_sync_bound = true;

    const ids = ['input_preview', 'output_image', 'groundtruth_preview'];
    const state = { scale: 1, x: 0, y: 0, dragging: false, lastX: 0, lastY: 0 };

    const clamp = (v, lo, hi) => Math.min(hi, Math.max(lo, v));

    const roots = () => ids.map((id) => document.getElementById(id)).filter(Boolean);
    const targets = () => roots().map((r) => r.querySelector('img')).filter(Boolean);

    const apply = () => {
      const t = `translate(${state.x}px, ${state.y}px) scale(${state.scale})`;
      for (const el of targets()) {
        el.style.transform = t;
        el.style.transformOrigin = '0 0';
      }
    };

    const beginDrag = (e) => {
      state.dragging = true;
      state.lastX = e.clientX;
      state.lastY = e.clientY;
    };

    const onMove = (e) => {
      if (!state.dragging) return;
      state.x += e.clientX - state.lastX;
      state.y += e.clientY - state.lastY;
      state.lastX = e.clientX;
      state.lastY = e.clientY;
      apply();
    };

    const endDrag = () => {
      state.dragging = false;
    };

    const onWheel = (e) => {
      if (!e.ctrlKey && !e.metaKey && !e.shiftKey) e.preventDefault();
      state.scale = clamp(state.scale * (e.deltaY < 0 ? 1.1 : 0.9), 0.5, 12);
      apply();
    };

    const bind = () => {
      for (const root of roots()) {
        if (root.dataset.syncBound === '1') continue;
        root.dataset.syncBound = '1';
        root.style.cursor = 'grab';
        root.addEventListener('mousedown', beginDrag);
        root.addEventListener('wheel', onWheel, { passive: false });
      }
      apply();
    };

    window.addEventListener('mousemove', onMove);
    window.addEventListener('mouseup', endDrag);

    bind();
    setInterval(bind, 700);
  } catch (err) {
    console.warn('sync_js disabled:', err);
  }
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown('## Real-ESRGAN Pytorch')

    input_image = gr.Image(type='pil', label='Upload Image', sources=['upload'])
    groundtruth_image = gr.Image(type='pil', label='Upload Ground Truth', sources=['upload'])

    input_model_image = gr.Radio(
        [('x2', 2), ('x4', 4), ('x8', 8)],
        type='value',
        value=4,
        label='Model Upscale/Enhance Type'
    )

    submit_button = gr.Button('Submit')

    # Row 1: input preview + output image
    with gr.Row():
        input_preview = gr.Image(type='pil', label='Input Preview', elem_id='input_preview')
        output_image = gr.Image(type='pil', label='Output Image', elem_id='output_image')

    # Row 2: keep Ground Truth frame width equal to one frame above
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('')
        with gr.Column(scale=2):
            groundtruth_preview = gr.Image(type='pil', label='Ground Truth Preview', elem_id='groundtruth_preview')
        with gr.Column(scale=1):
            gr.Markdown('')

    submit_button.click(
        fn=infer_image_for_ui,
        inputs=[input_image, input_model_image, groundtruth_image],
        outputs=[input_preview, output_image, groundtruth_preview],
    )

    demo.load(fn=lambda: None, inputs=None, outputs=None, js=sync_js)


if __name__ == '__main__':
    demo.launch(debug=True, show_error=True, share=True)
'''

Path('app.py').write_text(app_code, encoding='utf-8')
print('app.py patched: equal frame size for all 3 images')

!python -m py_compile infer.py app.py
print('Patch complete. Run Cell 7 to launch app.')

/Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch
app.py updated from notebook cell 5
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://22790bb5e3f02fb1c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
^C
Traceback (most recent call last):
  File "/Users/quang/.pyenv/versions/3.10.4/lib/python3.10/site-packages/gradio/blocks.py", line 3109, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch/app.py", line 34, in <module>
    demo.launch(debug=True, show_error=True, share=True)
  File "/Users/quang/.pyenv/versions/3.10.4/lib/python3.10/site-packages/gradio/blocks.py", line 3016, in launch
    self.block_thread()
  File "/Users/quang/.

In [ ]:
# Cell 6: Repair current Kaggle runtime (no venv)
%cd /kaggle/working/Real-ESRGAN_Pytorch

# Fix sitecustomize dependency first
!python -m pip install -q --upgrade pip setuptools wheel wrapt

# Reinstall a compatible stack in current runtime
!python -m pip install -q --no-cache-dir --force-reinstall \
    "numpy==2.2.6" \
    "pandas==2.2.3" \
    "gradio==5.31.0" \
    "gradio-client==1.10.1"

# Quick ABI check
!python -c "import numpy, pandas, gradio, wrapt; print('OK ->', numpy.__version__, pandas.__version__, gradio.__version__, wrapt.__version__)"

print('Cell 6 done. Restart kernel once, then run Cell 7.')

Updated app: /Users/quang/Desktop/fpt/Real-ESRGAN_Pytorch/app.py
infer.py already has spaces fallback (no change needed).
Done. Run the app launch cell next.


In [ ]:
# Cell 7: Launch minimal stable app
%cd /kaggle/working/Real-ESRGAN_Pytorch

!python -c "import numpy, pandas, gradio, wrapt; print('Subprocess ->', numpy.__version__, pandas.__version__, gradio.__version__, wrapt.__version__)"
!python app.py --share